In [1]:
import os
os.chdir("../")

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: float
    params_learning_rate: float
    params_include_top: bool
    params_weights: str 
    params_classes:int 
    

In [4]:
from deepClassifier.constants import *
from deepClassifier.utils import read_yaml,create_directories

In [5]:
class ConfigurationManager:
    def __init__(self,
        config_file_path:CONFIG_FILE_PATH,
        params_file_path:PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
    
    def get_prepare_base_model_config(self)->PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        create_directories(config.root_dir)
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir = Path(config.root_dir),
            base_model_path = Path(config.base_model_path),
            updated_base_model_path = Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE
            params_learning_rate = self.params.LEARNING_RATE,
            params_include_top = self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )
        return prepare_base_model_config 


In [6]:
import tensorflow as tf
class PrepareBaseModel:
    def __init__(self,config:PrepareBaseModelConfig):
        self.config = config 

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape = self.config.params_image_size,
            weights = self.config.params_weights,
            include_top = self.config.params_include_top
        )
        base_model_path = self.config.base_model_path
        self.save_model(path=base_model_path,model=self.model)

    @staticmethod
    def _prepare_full_model(model,classes,freeze_all,freeze_till,learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable=False 
        elif (freeze_till is not None) and (freeze_till>0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable=False
        
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units = classes,
            activation = "softmax",

        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs= model.input,
            output = prediction
        )
        full_model.compile(
            
        )

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model = self.model,
            classes = self.config.params_classes,
            freeze_all= True,
            freeze_till = None,
            learning_rate = self.config.params_learning_rate
        )
    

    @staticmethod
    def save_model(path:Path,model:tf.keras.Model):
        model.save(path)

In [ ]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
    prepare_base_model.save_model()
except Exception as e:
    raise e 